# 🕸️ DeepFetch — Agent-Ready Web Engine

**Universal scraping & automation engine — running in your Colab runtime, exposed publicly via a secure tunnel.**

| Feature | Details |
|---|---|
| 🎭 Stealth Playwright | Anti-detection, fingerprint rotation |
| 🤖 AI Extraction | Multi-provider fallback (Groq / Gemini / OpenAI / Ollama…) |
| 🔑 Session Store | AES-256 encrypted cookie persistence |
| 📡 REST + WebSocket API | Full async job queue, live streaming |
| 🐍 Python Client | Drop-in agent SDK included below |

**Run all cells top-to-bottom.** After setup, your public URL + API key are printed at the end of Step 5.

> ⏱️ First run takes ~5 min (installs Node 22 + Playwright Chromium). Subsequent runs with saved runtime: ~60 sec.


In [ ]:
#@title ⚙️ Step 1 — Configuration (fill in any AI keys you want)
#@markdown AI keys are **optional** — DeepFetch works without them (Playwright only).
#@markdown With a key, AI extraction kicks in when selectors fail.

GROQ_API_KEY = "" #@param {type:"string"}
#@markdown > Free tier at console.groq.com — fast, recommended

GOOGLE_AI_KEY = "" #@param {type:"string"}
#@markdown > Free tier at aistudio.google.com

OPENAI_API_KEY = "" #@param {type:"string"}
ANTHROPIC_API_KEY = "" #@param {type:"string"}

DEEPFETCH_PORT = 3000 #@param {type:"integer"}
#@markdown ---
#@markdown ### Session storage (optional)
#@markdown Paste session cookies to enable authenticated scraping on restart.
INSTAGRAM_COOKIES_JSON = "" #@param {type:"string"}
print("✅ Config saved. Run the next cell.")


In [ ]:
# ╔══════════════════════════════════════════════╗
# ║  Step 2 — Install system dependencies       ║
# ║  Node.js 22 · Chromium · build tools        ║
# ╚══════════════════════════════════════════════╝
import subprocess, sys, os

def run(cmd, **kw):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    if result.returncode != 0:
        print(result.stdout[-2000:] if result.stdout else '')
        print(result.stderr[-2000:] if result.stderr else '')
        raise RuntimeError(f"Command failed: {cmd}")
    return result.stdout.strip()

print("📦 Installing Node.js 22...")
run("curl -fsSL https://deb.nodesource.com/setup_22.x | bash - 2>&1 | tail -5")
run("apt-get install -y nodejs 2>&1 | tail -5")
node_ver = run("node --version")
npm_ver  = run("npm --version")
print(f"✅ Node {node_ver}  npm {npm_ver}")

print("📦 Installing system packages (chromium, python3-pip, cloudflared)...")
# System deps for Playwright
run("apt-get install -y chromium-browser libnss3 libatk1.0-0 libatk-bridge2.0-0 "
    "libcups2 libdrm2 libxkbcommon0 libxcomposite1 libxdamage1 libxfixes3 "
    "libxrandr2 libgbm1 libasound2 libpango-1.0-0 2>&1 | tail -3")

# cloudflared — zero-auth public tunnel
run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 "
    "-O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared")
cf_ver = run("cloudflared --version")
print(f"✅ cloudflared {cf_ver}")
print("\n✅ System dependencies ready.")


In [ ]:
# ╔══════════════════════════════════════════════╗
# ║  Step 3 — Clone · Install · Build           ║
# ╚══════════════════════════════════════════════╝
import os

REPO_DIR = "/deepfetch"

if not os.path.exists(REPO_DIR):
    print("📥 Cloning ferelking242/deepfetch...")
    run(f"git clone --depth 1 https://github.com/ferelking242/deepfetch {REPO_DIR}")
else:
    print("📥 Pulling latest changes...")
    run(f"cd {REPO_DIR} && git pull --ff-only")

print("📦 Installing server dependencies...")
run(f"cd {REPO_DIR}/server && npm ci --prefer-offline 2>&1 | tail -5")

print("📦 Installing dashboard dependencies...")
run(f"cd {REPO_DIR}/dashboard && npm ci --prefer-offline 2>&1 | tail -5")

print("🎭 Installing Playwright Chromium...")
os.environ["PLAYWRIGHT_BROWSERS_PATH"] = "/ms-playwright"
run(f"cd {REPO_DIR}/server && npx playwright install chromium --with-deps 2>&1 | tail -8")
print("✅ Playwright ready")

print("🔨 Building TypeScript server...")
run(f"cd {REPO_DIR}/server && npm run build 2>&1")
print("✅ Server built")

print("🔨 Building React dashboard...")
run(f"cd {REPO_DIR}/dashboard && npm run build 2>&1 | tail -5")
print("✅ Dashboard built")
print("\n🎉 Build complete! Run the next cell.")


In [ ]:
# ╔══════════════════════════════════════════════╗
# ║  Step 4 — Generate configuration            ║
# ╚══════════════════════════════════════════════╝
import secrets, yaml, os

# install pyyaml if needed
try:
    import yaml
except ImportError:
    run("pip install pyyaml -q")
    import yaml

REPO_DIR = "/deepfetch"
CONFIG_PATH = f"{REPO_DIR}/config.yaml"
DATA_DIR = "/deepfetch-data"
os.makedirs(DATA_DIR, exist_ok=True)

master_secret = secrets.token_hex(32)
enc_key       = secrets.token_hex(32)

# Build providers list — include only those with keys
providers = [{"name": "ollama", "local": True, "model": "llama3.2", "base_url": "http://localhost:11434"}]
if GROQ_API_KEY:        providers.append({"name": "groq",      "api_key": GROQ_API_KEY,       "model": "llama-3.3-70b-versatile"})
if GOOGLE_AI_KEY:       providers.append({"name": "gemini",    "api_key": GOOGLE_AI_KEY,       "model": "gemini-2.0-flash"})
if OPENAI_API_KEY:      providers.append({"name": "openai",    "api_key": OPENAI_API_KEY,       "model": "gpt-4o-mini"})
if ANTHROPIC_API_KEY:   providers.append({"name": "anthropic", "api_key": ANTHROPIC_API_KEY,    "model": "claude-haiku-4-5"})

config = {
    "server":   {"port": DEEPFETCH_PORT, "host": "0.0.0.0", "master_secret": master_secret},
    "browser":  {"pool_max": 0, "pool_reserved": 1, "context_ttl_seconds": 300,
                 "navigation_timeout_ms": 30000, "headless": True},
    "resources":{"cpu_threshold_pct": 85, "ram_threshold_pct": 80},
    "queue":    {"max_retries": 3, "retry_base_delay_ms": 2000, "result_ttl_seconds": 86400},
    "ai_engine":{"enabled": True, "trigger": "on_selector_failure",
                 "max_html_chars": 50000, "timeout_ms": 15000, "providers": providers},
    "zeusdl":   {"binary": "zeusdl", "extra_flags": []},
    "sessions": {"encryption_key": enc_key, "check_interval_seconds": 1800},
    "data_dir": DATA_DIR,
}

with open(CONFIG_PATH, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"✅ Config written to {CONFIG_PATH}")
print(f"   AI providers active: {[p['name'] for p in providers]}")
print(f"   Data directory: {DATA_DIR}")

# Store secrets in module-level vars for later cells
MASTER_SECRET = master_secret
print("\n✅ Secrets generated. Run the next cell to start the server.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Step 5 — Launch DeepFetch + cloudflared tunnel             ║
# ║  Prints: PUBLIC_URL + API_KEY — use these in every cell     ║
# ╚══════════════════════════════════════════════════════════════╝
import subprocess, threading, time, re, os, json, requests

REPO_DIR  = "/deepfetch"
PORT      = DEEPFETCH_PORT
ENV       = {**os.environ, "PLAYWRIGHT_BROWSERS_PATH": "/ms-playwright", "NODE_ENV": "production"}

# ── 1. Kill any leftover processes ──────────────────────────────
subprocess.run("pkill -f 'node.*index.js' 2>/dev/null; pkill -f cloudflared 2>/dev/null",
               shell=True, capture_output=True)
time.sleep(1)

# ── 2. Start DeepFetch server ────────────────────────────────────
print("🚀 Starting DeepFetch server...")
srv_log = open("/tmp/deepfetch.log", "w")
server_proc = subprocess.Popen(
    ["node", f"{REPO_DIR}/server/dist/index.js"],
    env={**ENV, "DF_CONFIG": f"{REPO_DIR}/config.yaml"},
    stdout=srv_log, stderr=srv_log,
    cwd=REPO_DIR
)

# Wait for server to be ready (max 30s)
for i in range(30):
    time.sleep(1)
    try:
        r = requests.get(f"http://localhost:{PORT}/v1/health", timeout=2)
        if r.status_code == 200:
            print(f"   ✅ Server up after {i+1}s")
            break
    except:
        pass
else:
    log = open('/tmp/deepfetch.log').read()[-3000:]
    print("❌ Server failed to start. Logs:")
    print(log)
    raise RuntimeError("DeepFetch server did not start")

# ── 3. Create first API key ──────────────────────────────────────
print("🔑 Creating API key...")
r = requests.post(
    f"http://localhost:{PORT}/v1/keys",
    json={"name": "colab-agent", "master_secret": MASTER_SECRET},
    timeout=10
)
if r.status_code not in (200, 201):
    raise RuntimeError(f"Key creation failed: {r.text}")
API_KEY = r.json().get("key") or r.json().get("api_key") or r.json()["data"]["key"]
print(f"   ✅ API key: {API_KEY}")

# ── 4. Start cloudflared tunnel ──────────────────────────────────
print("🌐 Starting cloudflared tunnel...")
PUBLIC_URL = None
cf_log = open("/tmp/cloudflared.log", "w")
cf_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"],
    stdout=cf_log, stderr=subprocess.STDOUT,
)

# Parse the public URL from cloudflared output
for _ in range(30):
    time.sleep(1)
    try:
        log_content = open("/tmp/cloudflared.log").read()
        match = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', log_content)
        if match:
            PUBLIC_URL = match.group(0)
            break
    except:
        pass

if not PUBLIC_URL:
    print("⚠️  cloudflared URL not found — falling back to localhost")
    PUBLIC_URL = f"http://localhost:{PORT}"

# ── 5. Print summary ─────────────────────────────────────────────
print()
print("═" * 60)
print("  🎉  DeepFetch is LIVE")
print("═" * 60)
print(f"  🌐  Public URL : {PUBLIC_URL}")
print(f"  🔑  API Key    : {API_KEY}")
print(f"  📊  Dashboard  : {PUBLIC_URL}/dashboard")
print(f"  📖  API Docs   : {PUBLIC_URL}/docs")
print("═" * 60)
print()
print("Copy PUBLIC_URL and API_KEY — they are already set as variables.")
print("Continue to the next cell for the Python client.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  DeepFetch Python Client — Agent SDK                        ║
# ║  Optimized for LLM agents: clean JSON, typed returns,       ║
# ║  sync + async, streaming, batch, sessions                   ║
# ╚══════════════════════════════════════════════════════════════╝
import requests, json, time, sys
from typing import Optional, Union
from dataclasses import dataclass, field

@dataclass
class ScrapeResult:
    url: str
    platform: str
    data: dict
    markdown: Optional[str] = None
    html: Optional[str] = None
    screenshot_b64: Optional[str] = None
    extracted_by: str = "unknown"
    duration_ms: int = 0
    job_id: Optional[str] = None

    def text(self) -> str:
        """Best available text representation for agent consumption."""
        if self.markdown:
            return self.markdown
        if self.data:
            return json.dumps(self.data, ensure_ascii=False, indent=2)
        return ""

    def __repr__(self):
        keys = list(self.data.keys()) if self.data else []
        return (f"ScrapeResult(platform={self.platform!r}, extracted_by={self.extracted_by!r}, "
                f"keys={keys}, duration={self.duration_ms}ms)")


class DeepFetchClient:
    """
    Agent-optimized Python client for DeepFetch.

    Usage:
        df = DeepFetchClient(PUBLIC_URL, API_KEY)
        result = df.scrape("https://www.youtube.com/watch?v=dQw4w9WgXcQ")
        print(result.text())            # best text for agent
        print(result.data)              # structured JSON
    """

    def __init__(self, base_url: str, api_key: str, timeout: int = 120):
        self.base = base_url.rstrip("/")
        self.headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
        self.timeout = timeout
        self.session = requests.Session()
        self.session.headers.update(self.headers)

    # ── Core scrape ────────────────────────────────────────────────
    def scrape(
        self,
        url: str,
        *,
        output: list[str] = ["json", "markdown"],
        session_id: Optional[str] = None,
        priority: str = "normal",
        scroll: bool = False,
        wait_for: Optional[str] = None,
        max_comments: int = 20,
        extract: Optional[list[str]] = None,
        timeout_ms: Optional[int] = None,
        screenshot: bool = False,
    ) -> ScrapeResult:
        """Scrape a URL and wait for the result (synchronous)."""
        if screenshot and "screenshot" not in output:
            output = list(output) + ["screenshot"]
        body = {
            "url": url, "sync": True, "output": output,
            "priority": priority, "session_id": session_id,
            "options": {
                "scroll": scroll, "max_comments": max_comments,
                **(({"wait_for": wait_for}) if wait_for else {}),
                **(({"extract": extract}) if extract else {}),
                **(({"timeout_ms": timeout_ms}) if timeout_ms else {}),
            }
        }
        r = self.session.post(f"{self.base}/v1/scrape", json=body, timeout=self.timeout)
        r.raise_for_status()
        return self._parse_result(r.json())

    def scrape_text(self, url: str, **kw) -> str:
        """Scrape and return the best text for agent consumption."""
        return self.scrape(url, **kw).text()

    def scrape_json(self, url: str, **kw) -> dict:
        """Scrape and return structured JSON data."""
        return self.scrape(url, output=["json"], **kw).data

    def scrape_markdown(self, url: str, **kw) -> str:
        """Scrape and return clean Markdown (great for LLM context)."""
        result = self.scrape(url, output=["markdown"], **kw)
        return result.markdown or result.text()

    # ── Async / polling ────────────────────────────────────────────
    def scrape_async(self, url: str, **kw) -> str:
        """Submit a scrape job and return job_id immediately (non-blocking)."""
        body = {"url": url, "sync": False, **kw}
        r = self.session.post(f"{self.base}/v1/scrape", json=body, timeout=30)
        r.raise_for_status()
        return r.json()["job_id"]

    def wait_for_job(self, job_id: str, poll_interval: float = 0.5) -> ScrapeResult:
        """Poll a job until done and return the result."""
        deadline = time.time() + self.timeout
        while time.time() < deadline:
            job = self.job(job_id)
            status = job.get("status")
            if status == "done":
                return self._parse_result(job)
            if status in ("failed", "cancelled"):
                raise RuntimeError(f"Job {job_id} {status}: {job.get('error')}")
            time.sleep(poll_interval)
        raise TimeoutError(f"Job {job_id} timed out after {self.timeout}s")

    # ── Batch ──────────────────────────────────────────────────────
    def batch(
        self,
        urls: list[str],
        output: list[str] = ["json", "markdown"],
        wait: bool = True,
        max_workers: int = 4,
    ) -> Union[list[str], list[ScrapeResult]]:
        """Scrape multiple URLs. Returns results if wait=True, job IDs if wait=False."""
        r = self.session.post(f"{self.base}/v1/batch",
                              json={"urls": urls, "output": output},
                              timeout=60)
        r.raise_for_status()
        job_ids = r.json().get("job_ids", [])
        if not wait:
            return job_ids
        results = []
        for jid in job_ids:
            try:
                results.append(self.wait_for_job(jid))
            except Exception as e:
                results.append({"job_id": jid, "error": str(e)})
        return results

    # ── Crawl ──────────────────────────────────────────────────────
    def crawl(
        self,
        url: str,
        max_pages: int = 10,
        same_domain: bool = True,
        output: list[str] = ["json", "markdown"],
        wait: bool = True,
    ) -> Union[str, list[ScrapeResult]]:
        """Recursively crawl a domain."""
        r = self.session.post(f"{self.base}/v1/crawl",
                              json={"url": url, "max_pages": max_pages,
                                    "same_domain": same_domain, "output": output},
                              timeout=60)
        r.raise_for_status()
        data = r.json()
        if not wait:
            return data.get("crawl_id")
        crawl_id = data.get("crawl_id")
        # Poll for all jobs
        deadline = time.time() + self.timeout * max_pages
        while time.time() < deadline:
            jobs = self.session.get(f"{self.base}/v1/jobs",
                                    params={"crawl_id": crawl_id}).json()
            pending = [j for j in jobs if j["status"] in ("queued", "running")]
            if not pending:
                return [self._parse_result(j) for j in jobs if j["status"] == "done"]
            time.sleep(1)
        raise TimeoutError("Crawl timed out")

    # ── Sessions ───────────────────────────────────────────────────
    def create_session(
        self,
        platform: str,
        cookies: Optional[list[dict]] = None,
        username: Optional[str] = None,
        password: Optional[str] = None,
    ) -> dict:
        """Create an authenticated session (cookies or credentials)."""
        payload = {"platform": platform}
        if cookies:
            payload["type"] = "cookies"
            payload["cookies"] = cookies
        else:
            payload["type"] = "credentials"
            payload["username"] = username
            payload["password"] = password
        r = self.session.post(f"{self.base}/v1/sessions", json=payload, timeout=60)
        r.raise_for_status()
        return r.json()

    def sessions(self) -> list[dict]:
        return self.session.get(f"{self.base}/v1/sessions", timeout=10).json()

    def check_session(self, session_id: str) -> dict:
        r = self.session.get(f"{self.base}/v1/sessions/{session_id}/check", timeout=30)
        r.raise_for_status()
        return r.json()

    # ── Jobs ───────────────────────────────────────────────────────
    def job(self, job_id: str) -> dict:
        return self.session.get(f"{self.base}/v1/jobs/{job_id}", timeout=10).json()

    def jobs(
        self,
        status: Optional[str] = None,
        platform: Optional[str] = None,
        limit: int = 50,
    ) -> list[dict]:
        params = {k: v for k, v in {"status": status, "platform": platform, "limit": limit}.items() if v}
        return self.session.get(f"{self.base}/v1/jobs", params=params, timeout=10).json()

    def cancel_job(self, job_id: str) -> dict:
        r = self.session.delete(f"{self.base}/v1/jobs/{job_id}", timeout=10)
        r.raise_for_status()
        return r.json()

    # ── System ─────────────────────────────────────────────────────
    def health(self) -> dict:
        """System status: CPU%, RAM%, pool size, queue depth."""
        return self.session.get(f"{self.base}/v1/health", timeout=10).json()

    def platforms(self) -> list[dict]:
        """List all supported platforms and their adapter status."""
        return self.session.get(f"{self.base}/v1/platforms", timeout=10).json()

    # ── Internal ───────────────────────────────────────────────────
    def _parse_result(self, raw: dict) -> ScrapeResult:
        result = raw.get("result") or raw
        return ScrapeResult(
            url=raw.get("url", result.get("url", "")),
            platform=raw.get("platform", result.get("platform", "generic")),
            data=result.get("data") or {},
            markdown=result.get("markdown"),
            html=result.get("html"),
            screenshot_b64=result.get("screenshot"),
            extracted_by=result.get("extracted_by", "unknown"),
            duration_ms=result.get("duration_ms", 0),
            job_id=raw.get("id") or raw.get("job_id"),
        )

# Instantiate immediately
df = DeepFetchClient(PUBLIC_URL, API_KEY)
h = df.health()
print("✅ DeepFetchClient ready")
print(f"   CPU: {h.get('cpu_pct', '?')}%  RAM: {h.get('ram_pct', '?')}%")
print(f"   Pool: {h.get('pool_size', '?')}/{h.get('pool_max', '?')}  Queue: {h.get('queue_depth', '?')} jobs")
print()
print("df is ready. Examples:")
print("  df.scrape_text('https://example.com')")
print("  df.scrape_json('https://www.reddit.com/r/Python.json')")
print("  df.scrape_markdown('https://docs.python.org')")
print("  df.batch(['https://a.com', 'https://b.com'])")
print("  df.health()")


## 📖 API Quick Reference

### Python Client (`df`)

| Method | Description |
|--------|-------------|
| `df.scrape(url, output=['json','markdown'], scroll=False, screenshot=False)` | Scrape & wait (sync) |
| `df.scrape_text(url)` | Best text for agent |
| `df.scrape_json(url)` | Structured data dict |
| `df.scrape_markdown(url)` | Clean markdown |
| `df.scrape_async(url)` → `job_id` | Submit, don't wait |
| `df.wait_for_job(job_id)` | Poll until done |
| `df.batch(urls, wait=True)` | Multiple URLs |
| `df.crawl(url, max_pages=10)` | Recursive crawl |
| `df.create_session(platform, cookies=[...])` | Authenticated scraping |
| `df.health()` | Server status |
| `df.jobs(status='running')` | List jobs |

### REST API (curl)

```bash
# Sync scrape
curl -X POST $PUBLIC_URL/v1/scrape \
  -H "Authorization: Bearer $API_KEY" \
  -H "Content-Type: application/json" \
  -d '{"url": "https://example.com", "sync": true, "output": ["json", "markdown"]}'

# Async scrape → poll
JOB=$(curl -s -X POST $PUBLIC_URL/v1/scrape -H "Authorization: Bearer $API_KEY" \
  -d '{"url": "https://..."}' | python3 -c "import sys,json; print(json.load(sys.stdin)['job_id'])")
curl $PUBLIC_URL/v1/jobs/$JOB

# Batch
curl -X POST $PUBLIC_URL/v1/batch \
  -H "Authorization: Bearer $API_KEY" \
  -d '{"urls": ["https://a.com", "https://b.com"]}'

# Health
curl $PUBLIC_URL/v1/health
```

### `ScrapeResult` object

```python
result.url           # str  — original URL
result.platform      # str  — 'youtube' | 'reddit' | 'generic' | …
result.data          # dict — structured JSON extraction
result.markdown      # str? — clean Markdown (LLM-ready)
result.html          # str? — raw HTML
result.screenshot_b64 # str? — base64 PNG (if output=['screenshot'])
result.extracted_by  # 'selectors' | 'ai' | 'blobs'
result.text()        # best text for agent — markdown if available, else JSON
```


In [ ]:
# ──────────────────────────────────────────────
# Demo 1 — Generic website scrape
# ──────────────────────────────────────────────
result = df.scrape("https://httpbin.org/json", output=["json", "markdown"])
print(result)
print()
print(result.text()[:1000])


In [ ]:
# ──────────────────────────────────────────────
# Demo 2 — Reddit (public API)
# ──────────────────────────────────────────────
result = df.scrape("https://www.reddit.com/r/MachineLearning/top.json?limit=5",
                   output=["json"])
print(result)
# Show post titles
posts = result.data.get("data", {}).get("children", [])
for p in posts[:5]:
    d = p.get("data", {})
    print(f"  • {d.get('title', '?')} ({d.get('score', 0)} pts)")


In [ ]:
# ──────────────────────────────────────────────
# Demo 3 — Batch scrape
# ──────────────────────────────────────────────
urls = [
    "https://httpbin.org/html",
    "https://httpbin.org/json",
    "https://httpbin.org/robots.txt",
]
results = df.batch(urls, output=["markdown"])
for r in results:
    if isinstance(r, ScrapeResult):
        print(f"✅ {r.url} — {r.extracted_by} — {r.duration_ms}ms")
        print(r.text()[:200])
        print()
    else:
        print(f"❌ Error: {r}")


In [ ]:
# ──────────────────────────────────────────────
# Demo 4 — Authenticated scraping (optional)
# Only run if you have cookies (from Step 1 form)
# ──────────────────────────────────────────────
if INSTAGRAM_COOKIES_JSON:
    import json as _json
    cookies = _json.loads(INSTAGRAM_COOKIES_JSON)
    session = df.create_session("instagram", cookies=cookies)
    session_id = session["session_id"]
    print(f"✅ Session created: {session_id}")

    # Scrape with session
    result = df.scrape("https://www.instagram.com/natgeo/",
                       session_id=session_id, output=["json"])
    print(result)
else:
    print("ℹ️  No Instagram cookies configured. Skipping authenticated demo.")
    print("   Add cookies in Step 1 (INSTAGRAM_COOKIES_JSON) to enable.")


In [ ]:
# ──────────────────────────────────────────────
# System health + running jobs
# ──────────────────────────────────────────────
import json
print("=== Health ===")
print(json.dumps(df.health(), indent=2))
print()
print("=== Recent Jobs ===")
for j in df.jobs(limit=10):
    status_icon = {"done": "✅", "running": "🔄", "queued": "⏳", "failed": "❌"}.get(j.get("status"), "?")
    print(f"  {status_icon} [{j['platform']}] {j['url'][:60]} — {j['status']}")


In [ ]:
# ──────────────────────────────────────────────
# View server logs (last 50 lines)
# ──────────────────────────────────────────────
try:
    with open("/tmp/deepfetch.log") as f:
        lines = f.readlines()
    print("".join(lines[-50:]))
except FileNotFoundError:
    print("No server log found (server not started yet?)")
